# CNN V5.1 P5 runner (Google Colab)

This notebook runs the authorized P5 preparation phase with the same V5.1 implementation and frozen identities as the local runner. It does not train a model or open test/OOD waveforms. Keep the data and outputs in your Google Drive.

## Drive layout

Upload the `study-v5-1` workspace code (including `src/`, `config/`, `scripts/`, and `requirements-v51.txt`) and these inputs under one Drive folder:

```text
MyDrive/CNN_V5_1_P5/
  study-v5-1/                         # code workspace
  frozen/p3/                           # FROZEN.json, train/val/test/ood.csv
  p4/                                  # p4_summary.json, weight_fit.json
  inventory_v51.csv
  sources/ae/                          # registered AE source root
  sources/dd/                          # registered DD source root
  sources/itu/                         # registered ITU-ARIS source root
  sources/musan/                       # registered MUSAN source root
  outputs/                             # new P5 attempt output is created here
```

Copy the exact `p3` and `p4` outputs from the passing corpus-005/split-005 attempt. Do not substitute another split or edit the frozen manifests. Ensure each source folder preserves the inventoried relative paths. A GPU is not used by P5; the A-chk deterministic pair ranking is CPU-bound. Keep Colab connected until the run exits.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import datetime, subprocess, sys, uuid

DRIVE_ROOT = Path('/content/drive/MyDrive/CNN_V5_1_P5')
STUDY = DRIVE_ROOT / 'study-v5-1'
FROZEN = DRIVE_ROOT / 'frozen/p3'
P4 = DRIVE_ROOT / 'p4'
INVENTORY = DRIVE_ROOT / 'inventory_v51.csv'
SOURCES = {
    'ae': DRIVE_ROOT / 'sources/ae',
    'dd': DRIVE_ROOT / 'sources/dd',
    'itu': DRIVE_ROOT / 'sources/itu',
    'musan': DRIVE_ROOT / 'sources/musan',
}
OUT = DRIVE_ROOT / 'outputs' / ('p5-' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])

required = [STUDY / 'scripts/run_cnn_v51_p5_colab.py', STUDY / 'requirements-v51.txt',
            STUDY / 'config/v51.yaml', FROZEN / 'FROZEN.json',
            P4 / 'p4_summary.json', P4 / 'weight_fit.json', INVENTORY]
missing = [str(p) for p in required if not p.exists()] + [str(p) for p in SOURCES.values() if not p.is_dir()]
if missing:
    raise FileNotFoundError('Missing required Drive inputs:\n' + '\n'.join(missing))
OUT.parent.mkdir(parents=True, exist_ok=True)
print('New output directory:', OUT)
print('This attempt will not overwrite existing outputs.')

In [ ]:
# Install the exact preparation dependency pins from the uploaded workspace.
!{sys.executable} -m pip install -r "{STUDY / 'requirements-v51.txt'}"

In [ ]:
import shlex

cmd = [sys.executable, str(STUDY / 'scripts/run_cnn_v51_p5_colab.py'),
       '--study-root', str(STUDY),
       '--frozen-attempt', str(FROZEN),
       '--source-inventory', str(INVENTORY),
       '--p4-attempt', str(P4),
       '--output', str(OUT)]
for source, root in SOURCES.items():
    cmd += ['--source-root', f'{source}={root}']

print('Command:', shlex.join(cmd))
log_path = OUT.parent / (OUT.name + '.log')
with log_path.open('w', encoding='utf-8') as log:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    code = proc.wait()
print('Exit code:', code, '| log:', log_path)
if code != 0:
    raise RuntimeError(f'P5 runner stopped with exit code {code}; preserve this attempt and inspect its log/gate output.')

## After it finishes

Inspect the printed gate summary and the files in the new `outputs/p5-*` directory. Preserve a failed attempt as-is. P6 can only start after P5 reports `PASS`; a failed P5 gate must not be bypassed by editing the frozen split or fixed rules. The `.log` file is written alongside the output directory.